# 🎵 RimaBR - Buscador de Rimas em Português Brasileiro

Sistema de busca de rimas com análise fonética avançada para português brasileiro.

**Como usar este notebook:**
1. Execute as células em ordem (Shift+Enter)
2. Aguarde a geração do banco de dados (5-10 min)
3. Use a interface de busca ou os comandos Python

---

## 📦 Passo 1: Instalação e Configuração

In [ ]:
# Clone o repositório
!git clone https://github.com/guitorte/language-play-ptbr.git
%cd language-play-ptbr

In [ ]:
# Instale as dependências
!pip install -q Flask Flask-CORS
print("✅ Dependências instaladas!")

## 🗄️ Passo 2: Geração do Banco de Dados

⏱️ **Isso vai demorar 5-10 minutos.** Aguarde até ver "Total indexed: 320090"

In [ ]:
# Gerar banco de dados completo (320k+ palavras)
!python rimabr_cli.py load data/palavras_completas.txt --batch-size 5000

## 🔍 Passo 3: Buscar Rimas (Interface Simples)

In [ ]:
# Importar o sistema de busca
import sys
sys.path.insert(0, '/content/language-play-ptbr')

from src.database.rimabr_db import RimaBRDatabase
from IPython.display import display, HTML
import pandas as pd

# Inicializar banco de dados
db = RimaBRDatabase('data/rimabr.db')
print("✅ Sistema RimaBR carregado!")

In [ ]:
# Função de busca com interface bonita
def buscar_rimas(palavra, limite=20, filtros=None):
    """
    Busca rimas para uma palavra
    
    Exemplos:
        buscar_rimas('amor')
        buscar_rimas('solidão', limite=30)
        buscar_rimas('paixão', filtros={'stress_type': 'oxítona'})
    """
    print(f"🔍 Buscando rimas para: {palavra}")
    print("=" * 80)
    
    resultados = db.search(palavra, filters=filtros, limit=limite)
    
    if not resultados:
        print("❌ Nenhuma rima encontrada.")
        return
    
    # Criar DataFrame para visualização
    dados = []
    for i, r in enumerate(resultados, 1):
        dados.append({
            'Rank': i,
            'Palavra': r['word'],
            'Score': f"{r['score']:.2f}",
            'Nível': r['level'].replace('_', ' ').title(),
            'Sílabas': r['features']['syllable_count'],
            'Tipo': r['features']['stress_type']
        })
    
    df = pd.DataFrame(dados)
    
    # Estilizar DataFrame
    styled = df.style.set_properties(**{
        'text-align': 'left',
        'font-size': '12px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#2563eb'), ('color', 'white'), ('font-weight', 'bold')]
    }]).hide(axis='index')
    
    display(styled)
    print(f"\n✅ Encontradas {len(resultados)} rimas para '{palavra}'")

print("✅ Função buscar_rimas() pronta para uso!")

## 🎯 Exemplos de Busca

In [ ]:
# Exemplo 1: Buscar rimas para "amor"
buscar_rimas('amor', limite=20)

In [ ]:
# Exemplo 2: Buscar rimas para "solidão"
buscar_rimas('solidão', limite=15)

In [ ]:
# Exemplo 3: Buscar rimas para "coração" (apenas oxítonas)
buscar_rimas('coração', limite=25, filtros={'stress_type': 'oxítona'})

In [ ]:
# Exemplo 4: Buscar rimas para "liberdade" (apenas paroxítonas)
buscar_rimas('liberdade', limite=20, filtros={'stress_type': 'paroxítona'})

## 🎨 Busca Personalizada

Use a célula abaixo para suas próprias buscas:

In [ ]:
# Digite sua palavra aqui:
buscar_rimas('paixão', limite=30)

## 📊 Estatísticas do Banco de Dados

In [ ]:
# Ver estatísticas do banco
!python rimabr_cli.py stats

## 🌐 Interface Web (Opcional - Requer Conta Ngrok)

**⚠️ ATENÇÃO:** A interface web via ngrok requer uma conta gratuita.

Se você quiser usar a interface web completa:

1. Crie uma conta grátis: https://dashboard.ngrok.com/signup
2. Pegue seu authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
3. Execute a célula abaixo com seu token

**Alternativa:** A interface Python acima já funciona perfeitamente! Use `buscar_rimas()` sem precisar da interface web.

In [ ]:
# OPCIONAL: Configurar ngrok (só se você quiser a interface web)

# 1. Instalar ngrok
!pip install -q pyngrok

# 2. Configure seu authtoken (pegue em: https://dashboard.ngrok.com/get-started/your-authtoken)
# Descomente a linha abaixo e coloque seu token:
# !ngrok config add-authtoken SEU_TOKEN_AQUI

print("✅ ngrok instalado!")
print("⚠️  Não esqueça de configurar o authtoken acima!")

In [ ]:
# OPCIONAL: Iniciar servidor Flask com ngrok
# Só execute se você configurou o authtoken acima!

import subprocess
import time

try:
    from pyngrok import ngrok
    
    # Iniciar Flask em background
    process = subprocess.Popen(
        ['python', 'webapp/api.py'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    
    # Aguardar servidor iniciar
    time.sleep(3)
    
    # Criar túnel ngrok
    public_url = ngrok.connect(5000)
    
    print("="*80)
    print("🎉 Interface Web RimaBR está rodando!")
    print("="*80)
    print(f"\n🌐 Acesse em: {public_url}")
    print("\n📱 Este link funciona em qualquer dispositivo!")
    print("\n⚠️  Para parar o servidor, execute: process.terminate()")
    print("="*80)
    
except Exception as e:
    print("❌ Erro ao iniciar interface web:")
    print(f"   {str(e)}")
    print("\n💡 Dica: Você configurou o authtoken do ngrok?")
    print("   1. Crie conta em: https://dashboard.ngrok.com/signup")
    print("   2. Pegue seu token em: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   3. Execute: !ngrok config add-authtoken SEU_TOKEN")
    print("\n✅ Mas não se preocupe! Use buscar_rimas() que funciona perfeitamente!")

In [ ]:
# Para parar o servidor (quando terminar)
# process.terminate()
# ngrok.disconnect(public_url)
# print("✅ Servidor parado")

## 📚 Busca por Padrão Fonético

In [ ]:
# Buscar palavras por padrão fonético
def buscar_por_padrao(filtros, limite=20):
    """
    Busca palavras por características fonéticas
    
    Filtros disponíveis:
    - stress_type: 'oxítona', 'paroxítona', 'proparoxítona'
    - syllable_count: número de sílabas
    - tonic_vowel: 'a', 'e', 'i', 'o', 'u'
    - suffix: terminação (ex: 'ção', 'dade', 'mente')
    
    Exemplo:
        buscar_por_padrao({'stress_type': 'oxítona', 'tonic_vowel': 'a'})
        buscar_por_padrao({'suffix': 'ção', 'syllable_count': 3})
    """
    resultados = db.search_by_pattern(filters=filtros, limit=limite)
    
    if not resultados:
        print("❌ Nenhuma palavra encontrada.")
        return
    
    print(f"🔍 Padrão: {filtros}")
    print("="*80)
    
    for i, r in enumerate(resultados, 1):
        f = r['features']
        print(f"{i:3}. {r['word']:<20} ({f['stress_type']}, {f['syllable_count']} sílabas)")
    
    print(f"\n✅ {len(resultados)} palavras encontradas")

# Exemplo: Palavras oxítonas com vogal tônica 'a'
buscar_por_padrao({'stress_type': 'oxítona', 'tonic_vowel': 'a'}, limite=30)

## 💡 Dicas de Uso

### Filtros Disponíveis:

```python
# Por tipo de acentuação
buscar_rimas('amor', filtros={'stress_type': 'oxítona'})

# Por número de sílabas
buscar_rimas('paixão', filtros={'syllable_count': 2})

# Por vogal tônica
buscar_rimas('alegria', filtros={'tonic_vowel': 'i'})

# Combinando filtros
buscar_rimas('coração', filtros={
    'stress_type': 'oxítona',
    'syllable_count': 3
})
```

### Contextos de Gênero:

Para ajustar o scoring por gênero musical, modifique o contexto:

```python
# Para Rap (aceita rimas mais flexíveis)
db_rap = RimaBRDatabase('data/rimabr.db', context='rap')

# Para Sertanejo (rimas tradicionais)
db_sertanejo = RimaBRDatabase('data/rimabr.db', context='sertanejo')
```

---

## 📖 Sobre o RimaBR

Sistema de busca de rimas com análise fonética avançada:

- **320.090 palavras** em português brasileiro
- **Análise fonética** completa (sílabas, tonicidade, vogais, consoantes)
- **Scoring multi-critério** (0-100) com explicações
- **Filtros avançados** por tipo de stress, sílabas, terminações
- **100% offline** - sem APIs externas

Desenvolvido com foco em composição musical, especialmente para gêneros brasileiros como Rap, Sertanejo, MPB e Repente.

**Repositório:** https://github.com/guitorte/language-play-ptbr